# Module 08: Object Serialization with Pickle
## Notebook 04: Advanced Serialization with Cloudpickle & Serialization Formats

Standard Python `pickle` fails on dynamic functions, lambdas, closures, and inner classes because it serializes functions **by reference (by module name and function name)** rather than serializing the actual function bytecode and enclosed variables.
In distributed computing (Spark, Ray, Dask, Celery) and advanced ML pipelines, **`cloudpickle`** bridges this gap by serializing Python code objects and closures directly.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand why standard `pickle` fails on lambdas and dynamically defined functions.
2. Serialize arbitrary functions, closures, and dynamically created classes using **`cloudpickle`**.
3. Understand how `cloudpickle` captures closure environments (`__closure__`).
4. Benchmark and compare modern serialization formats: **Pickle**, **JSON**, **Parquet**, and **Safetensors**.
5. Select the optimal serialization format based on security, language interoperability, and performance.

In [1]:
import os
import pickle
import cloudpickle
import json
import numpy as np

print(f"Cloudpickle Version: {cloudpickle.__version__}")

Cloudpickle Version: 3.1.2


### 1. The Standard Pickle Function Limitation
Standard `pickle` only serializes functions that exist at the top level of an importable module:
- It serializes only the string name of the module and the function name (`module.func_name`).
- It **cannot serialize lambdas** (`<lambda>` has no unique name).
- It **cannot serialize inner functions or closures** with captured local variables.

In [2]:
# 1. Lambda function
multiplier = lambda x: x * 3.14

try:
    pickle.dumps(multiplier)
except (pickle.PicklingError, AttributeError) as e:
    print(f"Standard Pickle on Lambda Failed as expected:\n -> {type(e).__name__}: {e}")

# 2. Closure capturing external environment
def make_normalizer(mean, std):
    # Inner function capturing local variables mean and std
    def normalize(val):
        return (val - mean) / std
    return normalize

norm_func = make_normalizer(mean=10.0, std=2.5)

try:
    pickle.dumps(norm_func)
except (pickle.PicklingError, AttributeError) as e:
    print(f"\nStandard Pickle on Closure Failed as expected:\n -> {type(e).__name__}: {e}")

Standard Pickle on Lambda Failed as expected:
 -> PicklingError: Can't pickle <function <lambda> at 0x7fb8fed5aca0>: attribute lookup <lambda> on __main__ failed

Standard Pickle on Closure Failed as expected:
 -> AttributeError: Can't get local object 'make_normalizer.<locals>.normalize'


### 2. Enter Cloudpickle: Serializing Code, Bytecode & Closures
`cloudpickle` inspects the function's underlying code object (`__code__`), serializes the raw Python bytecode, and recursively captures all referenced closure cells (`__closure__`):

In [3]:
# 1. Serializing lambda with cloudpickle
cloud_lambda_bytes = cloudpickle.dumps(multiplier)
restored_multiplier = cloudpickle.loads(cloud_lambda_bytes)
print(f"Restored Lambda Evaluation: multiplier(10) = {restored_multiplier(10):.2f}")

# 2. Serializing closure with cloudpickle
cloud_closure_bytes = cloudpickle.dumps(norm_func)
restored_norm = cloudpickle.loads(cloud_closure_bytes)
print(f"Restored Closure Evaluation: norm_func(15) = {restored_norm(15.0):.2f} (Expected: 2.00)")

Restored Lambda Evaluation: multiplier(10) = 31.40
Restored Closure Evaluation: norm_func(15) = 2.00 (Expected: 2.00)


### 3. Serializing Dynamically Generated Classes in Machine Learning
In dynamic ML workflows (e.g. factory patterns producing specialized custom loss functions or layer classes):
- Standard pickle fails because the dynamic class is not in `sys.modules`.
- `cloudpickle` serializes the class definition and its methods completely.

In [4]:
def create_polynomial_feature_extractor(degree):
    # Dynamically generated class inside a factory function
    class PolynomialTransformer:
        def __init__(self, deg=degree):
            self.degree = deg

        def transform(self, x):
            return np.column_stack([x**d for d in range(1, self.degree + 1)])

    return PolynomialTransformer()

extractor = create_polynomial_feature_extractor(degree=3)
test_vec = np.array([2.0, 3.0, 4.0])
print("Original Extractor Features:\n", extractor.transform(test_vec))

# Serialize dynamic class instance via cloudpickle
serialized_extractor = cloudpickle.dumps(extractor)
restored_extractor = cloudpickle.loads(serialized_extractor)
print("\nRestored Dynamic Class Features:\n", restored_extractor.transform(test_vec))

Original Extractor Features:
 [[ 2.  4.  8.]
 [ 3.  9. 27.]
 [ 4. 16. 64.]]

Restored Dynamic Class Features:
 [[ 2.  4.  8.]
 [ 3.  9. 27.]
 [ 4. 16. 64.]]


### 4. Machine Learning Serialization Architecture Matrix
| Format | Language Interoperability | Security | Zero-Copy Support | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Pickle / Cloudpickle** | Python only | Low (Unsafe) | Yes (Protocol 5) | Python-to-Python distributed workers, short-term RPC, Scikit-Learn pipelines. |
| **JSON** | Universal (C++, JS, Go) | High (Safe) | No (Text) | Configs, hyperparameter logs, human-readable metadata envelopes. |
| **Parquet** | Universal | High (Safe) | Yes (Arrow) | Tabular training datasets, columnar storage, analytics queries. |
| **Safetensors** | Universal (Rust, C++, Py) | High (Safe) | Yes (Zero-copy) | Deep learning model weights (LLMs, Diffusion models, PyTorch). |

In [5]:
# Benchmark Pickle vs JSON serialization on model telemetry
telemetry = {
    "model_name": "GradientBoostingClassifier",
    "params": {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.05},
    "feature_importances": [0.45, 0.25, 0.15, 0.10, 0.05]
}

pkl_bytes = pickle.dumps(telemetry)
json_bytes = json.dumps(telemetry).encode("utf-8")

print(f"Pickle Size: {len(pkl_bytes)} bytes")
print(f"JSON Size:   {len(json_bytes)} bytes")

Pickle Size: 198 bytes
JSON Size:   170 bytes
